# Customer Churn — Data Cleaning

## Objective

The objective of this stage is to prepare the customer dataset for reliable exploratory analysis.

Based on the data-understanding stage, the dataset contains several data-quality issues:

- Exact duplicate customer records
- Inconsistent capitalization in categorical values
- Missing values in selected columns
- Potential data-type inconsistencies

Each issue will be investigated and handled based on the business meaning of the data rather than applying a blanket cleaning rule.

## Cleaning Approach

1. Load the raw dataset
2. Remove validated duplicate records
3. Standardize categorical values
4. Investigate and handle missing values
5. Validate data types and category values
6. Perform final data-quality checks
7. Save the cleaned dataset for further analysis

In [2]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv("../data/raw/customer_churn.csv")

In [5]:
df.head()

,CustomerID,Gender,Age,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,C06889,Male,54.0,0,No,No,37,Yes,No,Fiber optic,...,No,No,Yes,No,Two year,No,Bank transfer,67.75,2661.46,No
1,C00415,Female,46.0,0,Yes,No,16,No,No phone service,DSL,...,No,No,No,No,Month-to-month,Yes,Electronic check,52.25,978.53,No
2,C03600,Female,34.0,0,No,No,0,Yes,No,DSL,...,No,Yes,No,No,One year,No,Electronic check,53.07,41.96,No
3,C01028,Female,52.0,0,No,No,23,Yes,Yes,Fiber optic,...,Yes,No,No,No,Month-to-month,Yes,Bank transfer,71.13,NaN,No
4,C04489,Female,29.0,0,No,No,16,Yes,No,No,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,33.48,597.84,No


In [30]:
df.shape

(7262, 22)

In [31]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMissing values:")
print(df.isna().sum())

Rows: 7262
Columns: 22

Duplicate rows: 12

Missing values:
CustomerID            0
Gender                0
Age                  87
SeniorCitizen         0
Partner               0
Dependents            0
Tenure                0
PhoneService          0
MultipleLines         0
InternetService       0
OnlineSecurity        0
OnlineBackup          0
DeviceProtection      0
TechSupport         102
StreamingTV           0
StreamingMovies       0
Contract              0
PaperlessBilling      0
PaymentMethod        88
MonthlyCharges       58
TotalCharges         87
Churn                 0
dtype: int64


In [12]:
duplicate_rows = df[df.duplicated(keep=False)].sort_values("CustomerID")

duplicate_rows

,CustomerID,Gender,Age,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
5482,C00386,Female,49.0,0,No,Yes,35,Yes,Yes,Fiber optic,...,Yes,Yes,No,No,Month-to-month,Yes,Mailed check,74.44,2370.01,Yes
5405,C00386,Female,49.0,0,No,Yes,35,Yes,Yes,Fiber optic,...,Yes,Yes,No,No,Month-to-month,Yes,Mailed check,74.44,2370.01,Yes
4919,C01058,Male,69.0,1,Yes,No,33,Yes,Yes,Fiber optic,...,Yes,No,No,No,Two year,No,Credit card,59.33,1822.60,No
6322,C01058,Male,69.0,1,Yes,No,33,Yes,Yes,Fiber optic,...,Yes,No,No,No,Two year,No,Credit card,59.33,1822.60,No
5995,C02386,Male,55.0,0,Yes,No,0,Yes,Yes,DSL,...,No,No,Yes,No,Month-to-month,Yes,Credit card,56.80,49.69,No
4030,C02386,Male,55.0,0,Yes,No,0,Yes,Yes,DSL,...,No,No,Yes,No,Month-to-month,Yes,Credit card,56.80,49.69,No
6503,C02789,Female,25.0,0,No,No,30,Yes,No,Fiber optic,...,Yes,Yes,No,No,One year,Yes,Bank transfer,70.17,1955.86,No
4811,C02789,Female,25.0,0,No,No,30,Yes,No,Fiber optic,...,Yes,Yes,No,No,One year,Yes,Bank transfer,70.17,1955.86,No
946,C05251,Male,37.0,0,Yes,No,28,Yes,No,DSL,...,No,Yes,No,No,Two year,Yes,Electronic check,43.31,1107.04,No
6751,C05251,Male,37.0,0,Yes,No,28,Yes,No,DSL,...,No,Yes,No,No,Two year,Yes,Electronic check,43.31,1107.04,No


In [32]:
duplicate_rows["CustomerID"].value_counts()

CustomerID
C00386    2
C01058    2
C02386    2
C02789    2
C05251    2
C06068    2
C06420    2
C06422    2
C06707    2
C06916    2
C06999    2
C07014    2
Name: count, dtype: int64

## Remove the duplicate copies

In [33]:
df = df.drop_duplicates()

In [34]:
# Check Duplicate Rows after removing duplicates.

print("Rows after removing duplicates:", df.shape[0])
print("Duplicate rows remaining:", df.duplicated().sum())

Rows after removing duplicates: 7250
Duplicate rows remaining: 0


## Standardize categorical values

In [35]:
categorical_columns = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts())


Partner
Partner
No     4085
Yes    3136
NO       16
yes      13
Name: count, dtype: int64

Dependents
Dependents
No     5600
Yes    1621
NO       21
yes       8
Name: count, dtype: int64

PhoneService
PhoneService
Yes    6524
No      697
yes      26
NO        3
Name: count, dtype: int64

PaperlessBilling
PaperlessBilling
Yes    4263
No     2958
yes      15
NO       14
Name: count, dtype: int64


In [36]:
for col in categorical_columns:
    df[col] = df[col].str.strip().str.lower().map({
        "yes": "Yes",
        "no": "No"
    })

.str.strip()
      ↓
remove accidental spaces

.str.lower()
      ↓
make everything lowercase

.map(...)
      ↓
convert standardized values back to our desired labels

In [37]:
for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts())


Partner
Partner
No     4101
Yes    3149
Name: count, dtype: int64

Dependents
Dependents
No     5621
Yes    1629
Name: count, dtype: int64

PhoneService
PhoneService
Yes    6550
No      700
Name: count, dtype: int64

PaperlessBilling
PaperlessBilling
Yes    4278
No     2972
Name: count, dtype: int64


## Missing Values

Missing value → investigate → understand business meaning → choose treatment.

In [38]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary[missing_summary["Missing_Count"] > 0]

missing_summary

,Missing_Count,Missing_Percentage
Age,87,1.20
TechSupport,102,1.41
PaymentMethod,87,1.20
MonthlyCharges,58,0.80
TotalCharges,87,1.20


# We will check missing values for the Age

In [39]:
df[df["Age"].isna()]["Churn"].value_counts()

Churn
No     66
Yes    21
Name: count, dtype: int64

In [40]:
df[df["Age"].isna()]["SeniorCitizen"].value_counts()

SeniorCitizen
0    79
1     8
Name: count, dtype: int64

In [41]:
df[df["Age"].isna()]["Gender"].value_counts()

Gender
Female    46
Male      41
Name: count, dtype: int64

The missingness is also only 1.20% of the dataset.

Therefore, for this project, median imputation is a reasonable approach.

# Check the median before imputing

In [25]:
age_median = df["Age"].median()

print("Age median:", age_median)

Age median: 43.0


In [26]:
df["Age"].isna().sum()

np.int64(87)

In [45]:
age_medians = df.groupby("SeniorCitizen")["Age"].median()

age_medians

SeniorCitizen
0    42.0
1    70.0
Name: Age, dtype: float64

In [46]:
df[df["Age"].isna()]["SeniorCitizen"].value_counts()

SeniorCitizen
0    79
1     8
Name: count, dtype: int64

# Fill the missing Age values

In [47]:
df["Age"] = df["Age"].fillna(
    df.groupby("SeniorCitizen")["Age"].transform("median")
)

print("Missing Age after imputation:", df["Age"].isna().sum())

Missing Age after imputation: 0


### Age Missing-Value Treatment

There were 87 missing values in the `Age` column.

Instead of using a single overall median, the `SeniorCitizen` variable was used to create more meaningful customer groups. The median age was calculated separately for each group:

- `SeniorCitizen = 0` → median age = 42
- `SeniorCitizen = 1` → median age = 70

The corresponding group median was used to impute missing Age values. This approach preserves the age distribution differences between senior and non-senior customers while avoiding the influence of extreme values that can affect mean-based imputation.

After imputation, no missing values remained in the `Age` column.

# We will check missing values for the MonthlyCharges

MonthlyCharges → 58 missing values

In [48]:
df[df["MonthlyCharges"].isna()].head()

,CustomerID,Gender,Age,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
392,C04621,Male,27.0,0,No,No,0,Yes,Yes,Fiber optic,...,No,NaN,Yes,Yes,Month-to-month,No,Mailed check,NaN,137.33,Yes
395,C05894,Male,51.0,0,No,No,63,Yes,No,Fiber optic,...,No,No,No,No,Month-to-month,No,Credit card,NaN,2695.57,No
427,C05213,Female,33.0,0,No,No,64,Yes,Yes,DSL,...,No,No,Yes,Yes,Two year,Yes,Bank transfer,NaN,3574.76,No
515,C03114,Female,18.0,0,No,No,47,Yes,Yes,DSL,...,Yes,No,Yes,Yes,Month-to-month,No,Mailed check,NaN,3020.50,No
1199,C00302,Male,43.0,0,No,No,36,Yes,Yes,No,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Electronic check,NaN,895.46,No


In [49]:
df[df["MonthlyCharges"].isna()]["Churn"].value_counts()

Churn
No     42
Yes    16
Name: count, dtype: int64

In [50]:
df[df["MonthlyCharges"].isna()]["InternetService"].value_counts()

InternetService
Fiber optic    31
DSL            21
No              6
Name: count, dtype: int64

In [51]:
df[df["MonthlyCharges"].isna()]["PhoneService"].value_counts()

PhoneService
Yes    52
No      6
Name: count, dtype: int64

# Investigate the relationship with services
Let's compare the normal MonthlyCharges distribution across InternetService and PhoneService.

In [52]:
df.groupby("InternetService")["MonthlyCharges"].agg(
    ["count", "min", "median", "mean", "max"]
)

,count,min,median,mean,max
InternetService,,,,,
DSL,2450,23.96,54.665,54.492653,89.98
Fiber optic,3448,38.42,69.325,69.486676,101.20
No,1294,18.00,27.620,28.078748,49.87


In [53]:
df.groupby("PhoneService")["MonthlyCharges"].agg(
    ["count", "min", "median", "mean", "max"]
)

,count,min,median,mean,max
PhoneService,,,,,
No,694,18.0,54.65,51.255893,90.74
Yes,6498,18.0,60.30,57.534529,101.20


# Check the missing rows more closely

In [54]:
df[df["MonthlyCharges"].isna()][[
    "InternetService",
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract"
]]

,InternetService,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract
392,Fiber optic,Yes,Yes,No,No,No,NaN,Yes,Yes,Month-to-month
395,Fiber optic,Yes,No,No,No,No,No,No,No,Month-to-month
427,DSL,Yes,Yes,No,No,No,No,Yes,Yes,Two year
515,DSL,Yes,Yes,Yes,No,Yes,No,Yes,Yes,Month-to-month
1199,No,Yes,Yes,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month
1273,DSL,Yes,No,No,No,No,No,No,Yes,Month-to-month
1522,Fiber optic,Yes,No,No,No,No,No,No,Yes,Month-to-month
1773,Fiber optic,Yes,Yes,Yes,No,No,No,Yes,No,Month-to-month
1868,Fiber optic,Yes,Yes,Yes,No,Yes,No,No,No,Month-to-month
1885,Fiber optic,Yes,No,No,Yes,No,No,Yes,No,Two year


# Our decision

Impute missing MonthlyCharges using the median MonthlyCharges within each InternetService group.

"InternetService is strongly associated with different MonthlyCharges distributions, so it provides useful information for estimating missing values."

In [55]:
monthly_charge_medians = df.groupby("InternetService")["MonthlyCharges"].median()

monthly_charge_medians

InternetService
DSL            54.665
Fiber optic    69.325
No             27.620
Name: MonthlyCharges, dtype: float64

In [56]:
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(
    df.groupby("InternetService")["MonthlyCharges"].transform("median")
)

In [57]:
print("Missing MonthlyCharges after imputation:", df["MonthlyCharges"].isna().sum())

Missing MonthlyCharges after imputation: 0


# We will check missing values for the TechSupport


In [58]:
df[df["TechSupport"].isna()]["InternetService"].value_counts()

InternetService
Fiber optic    48
DSL            30
No             24
Name: count, dtype: int64

In [59]:
pd.crosstab(
    df["InternetService"],
    df["TechSupport"],
    normalize="index"
) * 100

TechSupport,No,No internet service,Yes
InternetService,,,
DSL,68.127816,0.0,31.872184
Fiber optic,67.502186,0.0,32.497814
No,0.000000,100.0,0.000000


If InternetService = No, then TechSupport = No internet service 100% of the time.

That's not just a statistical pattern. It makes business sense.

A customer who doesn't have internet service cannot have normal technical support for an internet service.

In [60]:
df[df["TechSupport"].isna()][[
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "Contract",
    "Churn"
]].value_counts()

InternetService  OnlineSecurity       OnlineBackup         DeviceProtection     Contract        Churn
No               No internet service  No internet service  No internet service  Month-to-month  No       10
                                                                                One year        No        7
                                                                                Two year        No        6
DSL              No                   No                   No                   One year        No        5
Fiber optic      Yes                  No                   No                   Month-to-month  No        5
                 No                   Yes                  Yes                  Month-to-month  No        4
                                      No                   No                   Two year        No        4
                                                                                Month-to-month  No        3
DSL              Yes              

In [61]:
df.groupby("InternetService")["TechSupport"].agg(
    lambda x: x.mode()[0] if not x.mode().empty else np.nan
)

InternetService
DSL                             No
Fiber optic                     No
No             No internet service
Name: TechSupport, dtype: str

So now we can safely implement our cleaning rule.

| InternetService | Most common `TechSupport` |
| --------------- | ------------------------- |
| DSL             | **No**                    |
| Fiber optic     | **No**                    |
| No              | **No internet service**   |


In [ ]:
# Step 1 — Handle InternetService = No

df.loc[
    (df["InternetService"] == "No") & (df["TechSupport"].isna()),
    "TechSupport"
] = "No internet service"

# Find rows where InternetService is No and TechSupport is missing, and set TechSupport to No internet service.

In [ ]:
# Step 2 — Handle remaining missing values

df["TechSupport"] = df["TechSupport"].fillna(
    df.groupby("InternetService")["TechSupport"].transform(
        lambda x: x.mode()[0]
    )
)

# transform() puts the appropriate value onto each row, and fillna() uses it only where TechSupport is missing.

In [64]:
# Step 3 — Validate

print("Missing TechSupport after imputation:", df["TechSupport"].isna().sum())

Missing TechSupport after imputation: 0


# Why our TechSupport approach is strong

We didn't simply say "missing → fill with No".

We investigated the relationship:

1. No Internet Service
→ TechSupport = No internet service was already 100% among known values
→ Used this as a business rule.

2. DSL / Fiber optic
→ Both Yes and No were valid
→ Their group-wise mode was No
→ Used No only for the remaining missing values.

This is exactly the kind of reasoning you should be able to explain in an interview.

# Validate TechSupport

Before moving to PaymentMethod, let's make sure we didn't accidentally create an invalid category.

In [65]:
print(df["TechSupport"].value_counts(dropna=False))

TechSupport
No                     4057
Yes                    1893
No internet service    1300
Name: count, dtype: int64


In [66]:
print(df["TechSupport"].unique())

<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str


## PaymentMethod Missing Values

First understand why the data is missing → then choose the appropriate treatment.

In [67]:
df[df["PaymentMethod"].isna()]["Churn"].value_counts()

Churn
No     71
Yes    16
Name: count, dtype: int64

But we should not use Churn to decide the payment method.

Why?

Churn is our target/outcome variable. We are trying to understand whether payment method is associated with churn. If we use the outcome to manufacture the missing payment method, we could introduce bias into our analysis.

So we should investigate other customer/payment-related variables instead.

In [68]:
df[df["PaymentMethod"].isna()]["Contract"].value_counts()

Contract
Month-to-month    47
One year          21
Two year          19
Name: count, dtype: int64

In [69]:
df[df["PaymentMethod"].isna()]["PaperlessBilling"].value_counts()

PaperlessBilling
Yes    52
No     35
Name: count, dtype: int64

In [70]:
pd.crosstab(
    df["PaperlessBilling"],
    df["PaymentMethod"],
    normalize="index"
) * 100

PaymentMethod,Bank transfer,Credit card,Electronic check,Mailed check
PaperlessBilling,,,,
No,23.084780,21.109976,39.325843,16.479401
Yes,23.615712,22.006626,37.458590,16.919072


In [71]:
df["PaymentMethod"].value_counts(dropna=False)

PaymentMethod
Electronic check    2738
Bank transfer       1676
Credit card         1550
Mailed check        1199
NaN                   87
Name: count, dtype: int64

Electronic check is clearly the most common payment method.

We also investigated:

Churn → not appropriate for imputation because it is our outcome.
Contract → missing values occur across all three contract types.
PaperlessBilling → payment-method distributions are very similar between Yes and No, so it doesn't provide a strong inference rule.
Overall distribution → Electronic check is the most frequent category.

Therefore, simple mode imputation is reasonable here, especially because only 87 out of 7,250 (~1.2%) records are missing.

`Because the missing proportion is small and no strong business relationship was found to reliably infer the missing category, we use the most frequent payment method as a practical imputation.`

In [72]:
# Step 1 — Fill with mode

payment_mode = df["PaymentMethod"].mode()[0]

print("PaymentMethod mode:", payment_mode)

PaymentMethod mode: Electronic check


In [73]:
df["PaymentMethod"] = df["PaymentMethod"].fillna(payment_mode)

In [74]:
print(
    "Missing PaymentMethod after imputation:",
    df["PaymentMethod"].isna().sum()
)

Missing PaymentMethod after imputation: 0


## TotalCharges Missing Values

TotalCharges should generally increase as Tenure increases.

Customer A → Tenure = 2 months
Customer B → Tenure = 60 months

Giving both customers the same median TotalCharges would make little business sense.

# Step 1 — Check their tenure

In [75]:
df[df["TotalCharges"].isna()]["Tenure"].describe()

count    87.000000
mean     26.091954
std      18.273524
min       0.000000
25%      12.500000
50%      23.000000
75%      36.500000
max      72.000000
Name: Tenure, dtype: float64

In [76]:
df[df["TotalCharges"].isna()]["Tenure"].value_counts().sort_index()

Tenure
0     6
2     2
4     3
7     1
8     1
9     1
10    3
11    3
12    2
13    3
14    1
16    1
17    3
18    3
19    1
20    5
21    1
22    3
23    2
24    3
26    1
27    4
28    2
29    1
31    2
32    2
33    1
34    2
35    1
36    1
37    2
38    2
39    1
40    1
41    1
42    2
43    1
45    1
53    1
54    1
57    1
58    1
62    1
63    1
64    1
66    2
71    1
72    1
Name: count, dtype: int64

In [77]:
# Next investigation: compare TotalCharges with Tenure and MonthlyCharges

df[["Tenure", "MonthlyCharges", "TotalCharges"]].corr()

,Tenure,MonthlyCharges,TotalCharges
Tenure,1.000000,-0.026888,0.854619
MonthlyCharges,-0.026888,1.000000,0.395499
TotalCharges,0.854619,0.395499,1.000000


In [78]:
df[df["TotalCharges"].notna()][[
    "Tenure",
    "MonthlyCharges",
    "TotalCharges"
]].head(10)

,Tenure,MonthlyCharges,TotalCharges
0,37,67.75,2661.46
1,16,52.25,978.53
2,0,53.07,41.96
4,16,33.48,597.84
5,12,88.19,1149.51
6,43,73.70,3157.41
7,44,41.91,1906.26
8,29,65.68,2248.59
9,33,51.86,1775.89
10,23,82.59,2025.44


Tenure and TotalCharges have a strong positive relationship (0.855).

This makes business sense: customers who stay longer generally accumulate more charges.

MonthlyCharges also has a positive relationship with TotalCharges, but it is considerably weaker.

In [79]:
df.groupby("Tenure")["TotalCharges"].agg(
    ["count", "median", "mean"]
).head(20)

,count,median,mean
Tenure,,,
0,628,89.175,88.999013
1,77,136.110,132.626623
2,73,197.310,188.227397
3,58,268.990,264.304828
4,69,301.780,302.457246
5,96,375.295,368.015521
6,73,407.220,407.431096
7,74,466.365,459.679730
8,101,542.990,549.073366


In [80]:
df.groupby("Tenure")["TotalCharges"].count().describe()

count     73.000000
mean      98.123288
std       72.691308
min       14.000000
25%       69.000000
50%      100.000000
75%      125.000000
max      628.000000
Name: TotalCharges, dtype: float64

Median TotalCharges for each exact Tenure value.

Customer with missing TotalCharges
             ↓
Look at their Tenure
             ↓
Find median TotalCharges
for customers with that Tenure
             ↓
Use that median

We should calculate the tenure medians only from rows where TotalCharges is already known.
Because the missing values cannot be used to calculate their own replacement.

`Step 1 — Calculate tenure-wise medians`

In [81]:
total_charges_median = (
    df.groupby("Tenure")["TotalCharges"]
      .median()
)

In [82]:
print(total_charges_median.head(20))

Tenure
0       89.175
1      136.110
2      197.310
3      268.990
4      301.780
5      375.295
6      407.220
7      466.365
8      542.990
9      661.735
10     655.180
11     757.330
12     810.110
13     829.520
14     864.120
15     956.275
16    1009.850
17    1051.525
18    1194.005
19    1184.165
Name: TotalCharges, dtype: float64


`Step 2 — Use the mapping to fill missing values`

In [83]:
df["TotalCharges"] = df["TotalCharges"].fillna(
    df["Tenure"].map(total_charges_median)
)

`Step 3 — Validate`

In [84]:
print(
    "Missing TotalCharges after imputation:",
    df["TotalCharges"].isna().sum()
)

Missing TotalCharges after imputation: 0


### 🎯 Now: Final Missing-Value Audit

In [85]:
missing_values = df.isna().sum()

print(missing_values)

CustomerID          0
Gender              0
Age                 0
SeniorCitizen       0
Partner             0
Dependents          0
Tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [86]:
print("Total missing values:", df.isna().sum().sum())

Total missing values: 0


That means all 7,250 rows × 22 columns are currently free of missing values.

## Next: Data Type & Category Validation

Are the values stored in the correct data types, and are the categorical columns containing only valid business categories?

In [87]:
df.dtypes

CustomerID              str
Gender                  str
Age                 float64
SeniorCitizen         int64
Partner                 str
Dependents              str
Tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

In [88]:
df[["Age", "Tenure", "MonthlyCharges", "TotalCharges"]].describe()

,Age,Tenure,MonthlyCharges,TotalCharges
count,7250.000000,7250.000000,7250.000000,7250.000000
mean,43.083172,28.909241,56.950858,1713.333043
std,14.110705,19.119493,17.845678,1246.689799
min,18.000000,0.000000,18.000000,0.350000
25%,33.000000,14.000000,45.932500,743.627500
50%,43.000000,28.000000,59.800000,1507.690000
75%,53.000000,43.000000,69.815000,2486.430000
max,80.000000,72.000000,101.200000,7084.300000


In [89]:
print("Age below 18:", (df["Age"] < 18).sum())
print("Age above 80:", (df["Age"] > 80).sum())

print("Tenure below 0:", (df["Tenure"] < 0).sum())
print("Tenure above 72:", (df["Tenure"] > 72).sum())

print("MonthlyCharges <= 0:", (df["MonthlyCharges"] <= 0).sum())
print("TotalCharges <= 0:", (df["TotalCharges"] <= 0).sum())

Age below 18: 0
Age above 80: 0
Tenure below 0: 0
Tenure above 72: 0
MonthlyCharges <= 0: 0
TotalCharges <= 0: 0


A few important observations

Age

Minimum = 18
Maximum = 80
No impossible ages.
Our imputation didn't create anything outside the expected range.

Tenure

0–72 months is valid.
Remember: Tenure = 0 is not automatically an error. It can represent a very new customer.

MonthlyCharges

Minimum = 18
Maximum = 101.20
No zero or negative charges.

TotalCharges

Minimum = 0.35
Maximum = 7,084.30
No zero/negative values.
The very low value is not automatically invalid; a new/low-tenure customer can legitimately have low accumulated charges.

In [90]:
# Inspect Unique values for categorical columns

categorical_columns = [
    "Gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "Churn"
]

for col in categorical_columns:
    print(f"\n{col}:")
    print(df[col].unique())


Gender:
<StringArray>
['Male', 'Female']
Length: 2, dtype: str

Partner:
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

Dependents:
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

PhoneService:
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

MultipleLines:
<StringArray>
['No', 'No phone service', 'Yes']
Length: 3, dtype: str

InternetService:
<StringArray>
['Fiber optic', 'DSL', 'No']
Length: 3, dtype: str

OnlineSecurity:
<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

OnlineBackup:
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

DeviceProtection:
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

TechSupport:
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

StreamingTV:
<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

StreamingMovies:
<StringArray>
['No', 'No internet service', 'Yes']
Length: 3, dtype: str

Contract:
<StringArray>
['Two year', 'Month

## Final checks for Notebook 02

In [91]:
# Duplicate Check

print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [92]:
# Shape Check

print("Final shape:", df.shape)

Final shape: (7250, 22)


In [93]:
# CustomerID uniqueness

print("Unique CustomerIDs:", df["CustomerID"].nunique())
print("Total rows:", len(df))

Unique CustomerIDs: 7250
Total rows: 7250


Duplicate rows:       0
Final shape:          (7250, 22)
Unique CustomerIDs:   7250
Total rows:           7250

In [94]:
df.to_csv(
    "../data/processed/customer_churn_cleaned.csv",
    index=False
)

In [95]:
import os

print(
    os.path.exists(
        "../data/processed/customer_churn_cleaned.csv"
    )
)

True
